In [ ]:
! wget -N https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/lemma-corpus-small/lemmas_small.txt -P /tmp

In [ ]:
! wget -N https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/sentiment-corpus/default-sentiment-dict.txt -P /tmp

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/tmp'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install spark-nlp pyspark

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

from scipy import stats

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split, KFold, GroupKFold, GridSearchCV, StratifiedKFold

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsRegressor, KernelDensity, KDTree
from sklearn.metrics import *

In [ ]:
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession
from pyspark.sql.types import (StringType, BooleanType, IntegerType, FloatType, DateType, ArrayType, DoubleType) 
from pyspark.sql.functions import udf, col, lower, regexp_replace, size, array_join, split, shuffle, rand, when
import pyspark.sql.functions as funcs
from pyspark.sql.types import StringType
from pyspark.ml.feature import Tokenizer, StopWordsRemover, Word2Vec
from pyspark.ml.functions import array_to_vector
from pyspark.ml.classification import LogisticRegression, LinearSVC, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier, MultilayerPerceptronClassifier

import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag

In [ ]:
# Import Spark NLP
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline
import sparknlp

In [ ]:
# Start Spark Session with Spark NLP
# start() functions has 3 parameters: gpu, apple_silicon, and memory
# sparknlp.start(gpu=True) will start the session with GPU support
# sparknlp.start(apple_silicon=True) will start the session with macOS M1 & M2 support
# sparknlp.start(memory="16G") to change the default driver memory in SparkSession
spark = sparknlp.start()

In [ ]:
# Import the required modules and classes
from sparknlp.base import DocumentAssembler, Pipeline, Finisher

In [ ]:
from sparknlp.annotator import (
    SentenceDetector,
    Tokenizer,
    Lemmatizer,
    SentimentDetector
)

In [ ]:
import pyspark.sql.functions as F

### 1. Transforms raw texts to `document` annotation

In [ ]:
document_assembler = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

### 2. Sentence Detection

In [ ]:
sentence_detector = SentenceDetector().setInputCols(["document"]).setOutputCol("sentence")

### 3. Tokenization

In [ ]:
tokenizer = Tokenizer().setInputCols(["sentence"]).setOutputCol("token")

### 4: Lemmatization

In [ ]:
lemmatizer= Lemmatizer().setInputCols("token").setOutputCol("lemma") \
                        .setDictionary("/tmp/lemmas_small.txt", key_delimiter="->", value_delimiter="\t")

### 5: Sentiment Detection

In [ ]:
sentiment_detector= (
    SentimentDetector()
    .setInputCols(["lemma", "sentence"])
    .setOutputCol("sentiment_score")
    .setDictionary("/tmp/default-sentiment-dict.txt", ",")
)

### 6: Finisher

In [ ]:
finisher= (
    Finisher()
    .setInputCols(["sentiment_score"]).setOutputCols("sentiment")
)

In [ ]:
# Define the pipeline
pipeline = Pipeline(
    stages=[
        document_assembler,
        sentence_detector, 
        tokenizer, 
        lemmatizer, 
        sentiment_detector, 
        finisher
    ]
)

### Testing

In [ ]:
# Create a spark Data Frame with an example sentence
data = spark.createDataFrame(
    [
        [
            "The restaurant staff is really nice"
        ],
        [
            "I recommend others to avoid because it is too expensive"
        ]
    ]
).toDF("text") # use the column name `text` defined in the pipeline as input


In [ ]:
# Fit-transform to get predictions
result = pipeline.fit(data).transform(data).show(truncate = 50)

In [ ]:
# File location and type
file_location = "/kaggle/input/jgh-msss-global-exp-eng-only/sqllab_query_publicmsss_final_mtv_06172025_024124_20251112T192644.csv"
file_type = "csv"

In [ ]:
# CSV options
infer_schema = True
first_row_is_header = True
delimiter = ","

In [ ]:
df = spark.read \
.option("header", first_row_is_header) \
.csv(file_location)

In [ ]:
df.show(10, truncate = False)

In [ ]:
df = df.filter( (df.answervalue.isNotNull()) )
df.select('answervalue').show(10, truncate=False)

### Data Cleaning

In [ ]:
# Cleaning

@udf
def lower_clean_str(s):
    punc = "!''\"#$&\'()*+,-./:;<=>?@[\\]^_`{|}~"
    lowercased_str = s.lower()
    for c in punc:
        lowercased_str = lowercased_str.replace(c, '')
    return lowercased_str

In [ ]:
df = df.withColumn('text', lower_clean_str(df.answervalue))

In [ ]:
df.select('text').show(10, truncate=False)

### Calculate Sentiment Score and Assign Label

In [ ]:
# Create sentiment score
analyzer = SentimentIntensityAnalyzer()

In [ ]:
@udf
def pos_review(text):
    scores = analyzer.polarity_scores(text)
    if scores['pos'] > 0: return 1
    else: return 0

In [ ]:
df = df.withColumn('label', pos_review(df.text))
df = df.withColumn('label', df.label.cast('int'))
print(df.count())

In [ ]:
df.select(['text', 'label']).show(10, truncate=False)

In [ ]:
rating_count = df.groupBy("label").count().orderBy(col("count").desc())
rating_count.show(5)

In [ ]:
rating_count.rdd.map(lambda x:x.label).collect()

In [ ]:
plt.figure(figsize=(4,4))
plt.bar(rating_count.rdd.map(lambda x:x.label).collect(), rating_count.rdd.map(lambda x:x['count']).collect())
plt.title('Ratings distribution')
plt.ylabel('Count')
plt.xlabel('Positive Review')
plt.xticks(rating_count.rdd.map(lambda x:x.label).collect(), labels=['positive', 'negative'])

plt.show()

In [ ]:
cleaned_df = df.select(['text', 'label'])
print(cleaned_df.count())

In [ ]:
cleaned_df.show(10, truncate=False)

In [ ]:
cleaned_df.printSchema()

In [ ]:
cleaned_df = cleaned_df.withColumn("label", when(df.label == 1, "positive").otherwise("negative"))
cleaned_df.show(10, truncate=False)

In [ ]:
cleaned_df.printSchema()

In [ ]:
sentiment_detector = SentimentDetector() \
    .setInputCols(["lemma", "sentence"]) \
    .setOutputCol("sentiment_score") \
    .setDictionary("/tmp/default-sentiment-dict.txt", ",")

In [ ]:
pipeline = Pipeline(stages = [document_assembler,
                              sentence_detector,
                              tokenizer,
                              lemmatizer,
                              sentiment_detector,
                              finisher])

In [ ]:
empty_df = spark.createDataFrame([['']]).toDF("text")

In [ ]:
pipelineModel = pipeline.fit(cleaned_df)

In [ ]:
%%time
pipelineModel.transform(cleaned_df).show(1000, truncate=150)